# G1 Academy Bonus - Task 5: robot state, services & mode switching (using the wrapper)

## Introduction
Starting today you use the finished `sdk_wrapper.G1` wrapper directly, the same way Task 1 did -- you are **no longer** rebuilding native `LocoClient`/`RobotStateClient`/DDS calls by hand. There is no `ChannelFactoryInitialize`, no `Latest` subscriber class, no `ensure_channel_factory`: `G1.__init__` already owns the DDS ChannelFactory, every subscriber, and every SDK client for this kernel. This task covers reading robot state, listing/toggling services, and switching FSM modes safely.

**Using Codex/AI for this task:** every method below is a finished, documented method on `sdk_wrapper.G1`. If you get stuck, paste the method's docstring/signature (or the relevant line from `wrapper_cheatsheet.html`) into Codex and ask it to write the cell for you, then read what it produced before you run it against the robot. Knowing *what a call does* and *when it is safe to make it* is the point of this task, not typing it from memory.

In [ ]:
import sys
sys.path.append("..")
from sdk_wrapper import G1

# One G1 instance per kernel owns the DDS ChannelFactory, every subscriber, and every SDK
# client. Re-running this cell is fine (it reuses the module-level ChannelFactoryInitialize
# guard); constructing a second G1 with a different iface/domain_id in the same kernel raises.
g1 = G1(iface="eth0", domain_id=0)

## Task 1 - Read robot state: `get_lowstate()`, `get_odom()`, `get_battery()`, `get_state()`
- `g1.get_lowstate()` -- a snapshot dict of every motor (`q`, `dq`, `tau_est`, ...) and the IMU, taken from the latest `LowState_` message.
- `g1.get_odom()` -- the robot's current pose/twist from the odometry topic.
- `g1.get_battery()` -- a dict of BMS fields; the charge level is `battery["bms"]["soc"]` (percent).
- `g1.get_state()` -- the composite view: FSM `id`, `mode`, `motion_mode`, `gait`, plus battery, lowstate, service list, and SLAM info in one call. There is no separate `get_mode()`; `get_state()["id"]` / `get_state()["mode"]` is it.

In [ ]:
lowstate = g1.get_lowstate()
print("joints:", None if lowstate is None else len(lowstate["joint_positions"]))

battery = g1.get_battery()
print("battery soc:", None if battery is None else battery.get("bms", {}).get("soc"))

print("odom:", g1.get_odom())

state = g1.get_state()
print("state:", {k: state[k] for k in ("id", "mode", "motion_mode", "gait")})

## Task 2 - Services, and a live demo that `vui_service` gates audio
`vui_service` is the robot's **"Audio and Lighting Control Service"** -- `say()` and `set_headlight()` both go through it. This task proves that dependency end to end: read `vui_service`'s status (status `0` = ON, non-zero = OFF), turn it **off**, call `say()` to show audio no longer works, then turn it **back on** and confirm `say()` works again.

- `g1.get_service()` / `g1.get_service("vui_service")` -- list all services, or one row `{"name", "description", "status", "protected"}`.
- `g1.set_service(name, enabled)` -- explicit on/off (used here, so the off→on sequence is deterministic).
- `g1.toggle_service(name)` -- flips based on current status.

`g1.get_service()` calls the mainboard's `ServiceList` RPC, which intermittently returns code **3104 (RPC timeout)**; the wrapper then raises `RuntimeError("ServiceList failed: 3104")`. It is transient, so wrap every service call in a small retry.

In [ ]:
import time

def service_call_with_retry(fn, *args, retries=8, delay=2, **kwargs):
    """robot_state's ServiceList RPC intermittently returns 3104 (RPC timeout).
    Every g1 service call (get_service/set_service/toggle_service) can hit it,
    so retry a few times before giving up."""
    for attempt in range(1, retries + 1):
        try:
            return fn(*args, **kwargs)
        except RuntimeError as exc:
            print(f"service call attempt {attempt}/{retries} failed: {exc}")
            if attempt == retries:
                raise
            time.sleep(delay)

# 1) status before (0 = ON, non-zero = OFF)
before = service_call_with_retry(g1.get_service, "vui_service")
print("vui_service before:", before["status"])

# 2) turn the Audio & Lighting Control Service OFF
service_call_with_retry(g1.set_service, "vui_service", False)
time.sleep(3)  # give the service manager time to actually stop it
off = service_call_with_retry(g1.get_service, "vui_service")
print("vui_service now:", off["status"])

# 3) prove audio depends on it: say() should produce no sound with vui_service OFF
print("Testing say() with vui_service OFF (expect no audio):")
try:
    code = g1.say("If you can hear this, the voice service is still on.", language="EN")
    print("say() returned code:", code, "-- no sound expected while vui_service is off")
except Exception as exc:
    print("say() failed as expected without vui_service:", exc)

# 4) turn it back ON and confirm audio works again
service_call_with_retry(g1.set_service, "vui_service", True)
time.sleep(3)
after = service_call_with_retry(g1.get_service, "vui_service")
print("vui_service after:", after["status"])

print("Testing say() with vui_service ON (expect audio):")
print("say() returned code:", g1.say("Voice service restored. Audio works again.", language="EN"))

print("vui_service:", before["status"], "->", off["status"], "->", after["status"])

## Task 3 - Switch modes safely: `damp_mode()` / `prepare_mode()` / `walk_mode()` / `run_mode()` / `toggle_dev_mode()`
- `damp_mode()` -- FSM `1`: bounded joint damping, no locomotion. The always-available safe fallback -- call it before an emergency stop or before releasing controller ownership.
- `prepare_mode()` -- FSM `4`: the stand-up/ready pose, the usual step before `walk_mode()`.
- `walk_mode()` -- FSM `500` on this hardware (not `501` -- this academy's units run with the waist locked, only `WaistYaw` free).
- `run_mode()` -- FSM `802`.
- `toggle_dev_mode()` -- flips the `ai_sport` service; several Day 3 low-level control calls need dev mode enabled first.

Always go `damp_mode()` → `prepare_mode()` → `walk_mode()` in that order, one step at a time, checking `get_state()` between steps -- never jump straight to `run_mode()`.

In [ ]:
g1.damp_mode();    print("fsm:", g1.get_state()["id"])
g1.prepare_mode(); print("fsm:", g1.get_state()["id"])
g1.walk_mode();    print("fsm:", g1.get_state()["id"])

g1.toggle_dev_mode()
print("ai_sport:", g1.get_service("ai_sport"))

## Final part - build a mode-switching dashboard with Codex
You have now driven every FSM transition by hand from this notebook. The last exercise is to wrap those same `g1` calls in a small web dashboard -- **not** by writing it yourself, but by prompting Codex to build it against `sdk_wrapper.G1`.

A finished reference already runs on this robot: **`academy/visualizations/mode_control.py`, the mode-control dashboard served on port `8051`**. Open it in your browser first to see the target: one button per mode, a live state line, and buttons that grey out when a transition would be unsafe. Your job is to have Codex produce your own version on a *different* port so the two don't clash.

**Paste this prompt into Codex** (adjust to taste), then read the generated file before you run it against the robot:

> Using the `G1` class in `academy/sdk_wrapper.py`, write a Dash + dash-bootstrap-components web app `mode_switch_dashboard.py` that mirrors `academy/visualizations/mode_control.py` (the mode-control dashboard already running on port 8051). Requirements:
> - Construct a single `G1(iface="eth0", domain_id=0)` instance at startup and reuse it for every callback, guarded by a `threading.Lock`; never build more than one G1 per process.
> - One large button per FSM mode -- Zero Torque, Damp, Prepare, Walk, Run -- each calling the matching wrapper method: `g1.zero_torque_mode()`, `g1.damp_mode()`, `g1.prepare_mode()`, `g1.walk_mode()`, `g1.run_mode()`. Add a red **Stop** button wired to `g1.loco_stop()`.
> - A once-per-second `dcc.Interval` callback that reads `g1.get_state()` and shows the live `id` / `mode` / `motion_mode` / `gait` and battery soc in a status line, with the raw dict in a `<pre>` block.
> - Enforce the safe mode ladder: always allow Damp; disable Walk/Run unless the current `mode` is already prepare/walk/run. Wrap every robot call in try/except so one failed RPC never kills the app, and show the error text in the status line.
> - Bind to host `0.0.0.0` on port **8052** (a new port, so it does not clash with the 8051 dashboard) and print the URL on startup.

Use the cell below to launch or iterate on whatever Codex gives you.

In [ ]:
# Open-ended: run or refine the Codex-generated mode_switch_dashboard.py here.
# e.g. from a terminal:  python3 mode_switch_dashboard.py --port 8052
# then open http://<robot-ip>:8052 in your browser.

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.